# Zadanie samodzielne: Analiza Przestepczosci w Chicago

Analiza danych przy uzyciu PySpark na zbiorze *Chicago Crimes* (okolo 50 000 ostatnich zdarzen).

### Wymagania:
1. **Wczytanie i Czyszczenie Danych** - duplikaty, braki, bledne daty
2. **UDF i Pora Dnia** - klasyfikacja pory dnia z kolumny `date`
3. **Optymalizacja i Partycjonowanie** - `cache()`, broadcast join, zapis do Parquet z partycja po `year`
4. **Analiza i Plany Zapytan** - `.explain()` dla najciezszej agregacji
5. *(Opcjonalnie)* **MLlib** - model klasyfikujacy typ przestepstwa

Wzorujemy sie na DEMO Otomoto - ta sama struktura `SparkSession.builder`, `functions as F`, `groupBy().agg()`.

In [1]:
import os
# Jesli na macOS / homebrew java:
# os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType


In [2]:
spark = SparkSession.builder \
    .appName("Chicago Crimes") \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/24 13:55:39 WARN Utils: Your hostname, kali, resolves to a loopback address: 127.0.1.1; using 192.168.100.15 instead (on interface eth0)
26/05/24 13:55:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/24 13:55:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Wczytanie danych
# Plik chicago_crimes_sample.csv znajduje sie w materialy/lab_05/
CSV_PATH = "../../AAP_LAB_TASKS/materialy/lab_05/chicago_crimes_sample.csv"

df = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .csv(CSV_PATH)

df.printSchema()
print(f"Liczba wierszy: {df.count()}")
df.show(5)


26/05/24 13:55:44 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../../AAP_LAB_TASKS/materialy/lab_05/chicago_crimes_sample.csv.
java.io.FileNotFoundException: File ../../AAP_LAB_TASKS/materialy/lab_05/chicago_crimes_sample.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1Batch

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/eg/Desktop/AAP_LAB_TASKS/materialy/lab_05/chicago_crimes_sample.csv. SQLSTATE: 42K03

## 1. Czyszczenie danych
Usuwamy duplikaty, wiersze z brakami w kluczowych kolumnach i odfiltrowujemy bledne daty.

In [4]:
df_clean = df.dropDuplicates() \
    .dropna(subset=["date", "primary_type", "location_description"])

# Parsowanie daty (format ISO: 2026-05-07T00:00:00.000)
df_clean = df_clean.withColumn(
    "date_parsed",
    F.to_timestamp(F.col("date"), "yyyy-MM-dd'T'HH:mm:ss.SSS")
)
df_clean = df_clean.filter(F.col("date_parsed").isNotNull())

# Wyciagamy godzine i rok
df_clean = df_clean.withColumn("hour", F.hour("date_parsed")) \
                   .withColumn("year_int", F.year("date_parsed"))

print(f"Po czyszczeniu: {df_clean.count()} wierszy")
df_clean.select("date_parsed", "hour", "year_int", "primary_type").show(5)


NameError: name 'df' is not defined

## 2. UDF - Pora dnia
Klasyfikujemy pore dnia na podstawie godziny przestepstwa.

In [5]:
def time_of_day(h):
    if h is None:
        return "unknown"
    if 6 <= h < 12:
        return "rano"
    if 12 <= h < 18:
        return "dzien"
    if 18 <= h < 22:
        return "wieczor"
    return "noc"

time_of_day_udf = F.udf(time_of_day, StringType())

df_clean = df_clean.withColumn("pora_dnia", time_of_day_udf(F.col("hour")))
df_clean.select("hour", "pora_dnia").distinct().orderBy("hour").show()


NameError: name 'df_clean' is not defined

## 3. Optymalizacja - cache()
`cache()` przechowa DataFrame w pamieci - kolejne operacje beda znacznie szybsze.

In [6]:
df_clean.cache()
df_clean.count()  # trigger - dane zostaja faktycznie obliczone i zapisane w pamieci


NameError: name 'df_clean' is not defined

## 4. Analiza statystyczna

In [7]:
print("=== Top 10 typow przestepstw ===")
df_clean.groupBy("primary_type") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show(10, truncate=False)


=== Top 10 typow przestepstw ===


NameError: name 'df_clean' is not defined

In [8]:
print("=== Rozklad po porze dnia ===")
df_clean.groupBy("pora_dnia") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show()


=== Rozklad po porze dnia ===


NameError: name 'df_clean' is not defined

In [9]:
print("=== Top 10 lokalizacji przestepstw ===")
df_clean.groupBy("location_description") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show(10, truncate=False)


=== Top 10 lokalizacji przestepstw ===


NameError: name 'df_clean' is not defined

In [10]:
print("=== Pora dnia per typ przestepstwa (top 5 typow) ===")
df_clean.groupBy("primary_type", "pora_dnia") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show(15, truncate=False)


=== Pora dnia per typ przestepstwa (top 5 typow) ===


NameError: name 'df_clean' is not defined

## 5. Broadcast join
Mala tabela slownikowa z kategoriami przestepstw - rozsylamy ja do wszystkich executorow.

In [11]:
from pyspark.sql import Row

kategorie = spark.createDataFrame([
    Row(primary_type="THEFT",              kategoria="majatkowe"),
    Row(primary_type="BATTERY",            kategoria="osobowe"),
    Row(primary_type="ASSAULT",            kategoria="osobowe"),
    Row(primary_type="CRIMINAL DAMAGE",    kategoria="majatkowe"),
    Row(primary_type="NARCOTICS",          kategoria="narkotykowe"),
    Row(primary_type="BURGLARY",           kategoria="majatkowe"),
    Row(primary_type="MOTOR VEHICLE THEFT", kategoria="majatkowe"),
    Row(primary_type="ROBBERY",            kategoria="majatkowe"),
    Row(primary_type="DECEPTIVE PRACTICE", kategoria="majatkowe"),
])

df_kat = df_clean.join(F.broadcast(kategorie), on="primary_type", how="left")
df_kat.groupBy("kategoria").count().orderBy(F.col("count").desc()).show()


NameError: name 'df_clean' is not defined

## 6. Plan zapytania - .explain()

In [12]:
# Plan wykonania dla najciezszej agregacji - groupBy 2 kolumn + orderBy
hardest_query = df_clean.groupBy("primary_type", "pora_dnia") \
    .count() \
    .orderBy(F.col("count").desc())

hardest_query.explain()


NameError: name 'df_clean' is not defined

## 7. Spark SQL
Mozemy odpytywac DataFrame przez SQL - tak jak w demo Otomoto.

In [13]:
df_clean.createOrReplaceTempView("crimes")

spark.sql("""
    SELECT pora_dnia,
           primary_type,
           COUNT(*) AS ile
    FROM crimes
    WHERE pora_dnia = 'noc'
    GROUP BY pora_dnia, primary_type
    ORDER BY ile DESC
    LIMIT 10
""").show(truncate=False)


NameError: name 'df_clean' is not defined

## 8. Zapis do Parquet z partycjonowaniem po roku

In [ ]:
OUTPUT_PATH = "chicago_crimes_clean.parquet"

df_clean.select("id", "primary_type", "location_description",
                "year_int", "hour", "pora_dnia", "date_parsed") \
    .write \
    .mode("overwrite") \
    .partitionBy("year_int") \
    .parquet(OUTPUT_PATH)

print(f"Zapisano do {OUTPUT_PATH}")


In [ ]:
# Odczyt z powrotem - sprawdzamy partycje
df_back = spark.read.parquet(OUTPUT_PATH)
print(f"Wczytano {df_back.count()} rekordow.")
df_back.printSchema()


In [ ]:
spark.stop()
print("Spark zamkniety.")
